In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
# -----------------------------
# Hyperparameters
# -----------------------------
batch_size = 64
learning_rate = 0.01
num_epochs = 20


Using device: cuda


In [10]:
# -----------------------------
# CIFAR-10 Dataset
# -----------------------------
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_dataset = datasets.CIFAR10(root="./data", train=True,  transform=transform_train, download=True)
test_dataset  = datasets.CIFAR10(root="./data", train=False, transform=transform_test,  download=True)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

In [11]:
# -----------------------------
# Deeper LeNet-style CNN
#
# Stage 1: 3→64,  32×32 → 16×16 after pool
# Stage 2: 64→128, 16×16 → 8×8  after pool
# Stage 3: 128→256, 8×8 → 4×4   after pool
# Stage 4: 256→512, 4×4 → 2×2   after pool
# FC: 512×2×2 = 2048 → 256 → 10
#
# Each stage: one channel-expanding conv, then two same-channel convs with a skip
# -----------------------------
class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()

        # Stage 1: 3 → 64
        self.conv1  = nn.Conv2d(3,   64,  kernel_size=3, padding=1)
        self.bn1    = nn.BatchNorm2d(64)
        self.conv1a = nn.Conv2d(64,  64,  kernel_size=3, padding=1)
        self.bn1a   = nn.BatchNorm2d(64)
        self.conv1b = nn.Conv2d(64,  64,  kernel_size=3, padding=1)
        self.bn1b   = nn.BatchNorm2d(64)

        # Stage 2: 64 → 128
        self.conv2  = nn.Conv2d(64,  128, kernel_size=3, padding=1)
        self.bn2    = nn.BatchNorm2d(128)
        self.conv2a = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn2a   = nn.BatchNorm2d(128)
        self.conv2b = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn2b   = nn.BatchNorm2d(128)

        # Stage 3: 128 → 256
        self.conv3  = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3    = nn.BatchNorm2d(256)
        self.conv3a = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn3a   = nn.BatchNorm2d(256)
        self.conv3b = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn3b   = nn.BatchNorm2d(256)

        # Stage 4: 256 → 512
        self.conv4  = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn4    = nn.BatchNorm2d(512)
        self.conv4a = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.bn4a   = nn.BatchNorm2d(512)
        self.conv4b = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.bn4b   = nn.BatchNorm2d(512)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # FC: 512×2×2 = 2048 → 256 → 10
        self.fc1  = nn.Linear(512 * 2 * 2, 256)
        self.fc2  = nn.Linear(256, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Stage 1: 3×32×32 → 64×16×16
        x    = self.relu(self.bn1(self.conv1(x)))
        skip = x
        x    = self.relu(self.bn1a(self.conv1a(x)))
        x    = self.relu(self.bn1b(self.conv1b(x))) + skip
        x    = self.pool(x)

        # Stage 2: 64×16×16 → 128×8×8
        x    = self.relu(self.bn2(self.conv2(x)))
        skip = x
        x    = self.relu(self.bn2a(self.conv2a(x)))
        x    = self.relu(self.bn2b(self.conv2b(x))) + skip
        x    = self.pool(x)

        # Stage 3: 128×8×8 → 256×4×4
        x    = self.relu(self.bn3(self.conv3(x)))
        skip = x
        x    = self.relu(self.bn3a(self.conv3a(x)))
        x    = self.relu(self.bn3b(self.conv3b(x))) + skip
        x    = self.pool(x)

        # Stage 4: 256×4×4 → 512×2×2
        x    = self.relu(self.bn4(self.conv4(x)))
        skip = x
        x    = self.relu(self.bn4a(self.conv4a(x)))
        x    = self.relu(self.bn4b(self.conv4b(x))) + skip
        x    = self.pool(x)

        # FC: 512×2×2 → 256 → 10
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [12]:
model = LeNet5().to(device)
print(model)


LeNet5(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv1a): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv1b): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2a): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2b): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3a): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3b): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4a): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4b): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2,

In [13]:
# -----------------------------
# Loss and Optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(num_epochs):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")
    
    if (epoch + 1) % 2 == 0:
        torch.save(model.state_dict(), f"checkpoint_epoch_{epoch+1}.pt")
        print(f"Checkpoint saved at epoch {epoch+1}")


Epoch [1/20], Loss: 2.1732
Epoch [2/20], Loss: 1.8594
Checkpoint saved at epoch 2
Epoch [3/20], Loss: 1.6423
Epoch [4/20], Loss: 1.4945
Checkpoint saved at epoch 4
Epoch [5/20], Loss: 1.3975
Epoch [6/20], Loss: 1.3209
Checkpoint saved at epoch 6
Epoch [7/20], Loss: 1.2460
Epoch [8/20], Loss: 1.1709
Checkpoint saved at epoch 8
Epoch [9/20], Loss: 1.1043
Epoch [10/20], Loss: 1.0384
Checkpoint saved at epoch 10
Epoch [11/20], Loss: 0.9799
Epoch [12/20], Loss: 0.9305
Checkpoint saved at epoch 12
Epoch [13/20], Loss: 0.8819
Epoch [14/20], Loss: 0.8396
Checkpoint saved at epoch 14
Epoch [15/20], Loss: 0.8007
Epoch [16/20], Loss: 0.7627
Checkpoint saved at epoch 16
Epoch [17/20], Loss: 0.7336


Trajectory without batch norm

---

Epoch [1/20], Loss: 2.1732
Epoch [2/20], Loss: 1.8594
Checkpoint saved at epoch 2
Epoch [3/20], Loss: 1.6423
Epoch [4/20], Loss: 1.4945
Checkpoint saved at epoch 4
Epoch [5/20], Loss: 1.3975
Epoch [6/20], Loss: 1.3209
Checkpoint saved at epoch 6
Epoch [7/20], Loss: 1.2460
Epoch [8/20], Loss: 1.1709
Checkpoint saved at epoch 8
Epoch [9/20], Loss: 1.1043
Epoch [10/20], Loss: 1.0384
Checkpoint saved at epoch 10

In [ ]:
# Testing
# -----------------------------
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 94.15%
